# What a Corpus Can and Cannot Answer

**Computational Text Analysis Pack · Module 01 — Framing the Question**

*Technique:* Matching a literary or linguistic question to a countable method

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/00-foundations/01a_what_a_corpus_answers.ipynb)

Data: `masakhanews.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from collections import Counter

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['masakhanews.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('masakhanews.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Counting answers a narrow class of questions very well and another class not at
all. Knowing which is which saves weeks.

| Question | Countable? | Method |
|---|---|---|
| Does *corruption* appear more in politics reporting than sports? | **Yes** | keyness, 04a |
| What words surround *corruption* when it appears? | **Yes** | KWIC, 05a |
| Do these two writers use function words differently? | **Yes** | stylometry, 09b |
| Is this article *ironic* about corruption? | **No** | close reading |
| Does this article *approve* of the government? | **Barely** | sentiment, with heavy caveats |

The third column is the whole pack. The fourth row is why the pack does not
replace you.

## The tooling

No library does this notebook's work: deciding what a corpus can answer is
judgement, not computation. **pandas** is used only to look at what we have.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. It does not
    match combining marks, and a tone mark on ọ or ẹ is always one, so the
    pattern adds the combining-accent range U+0300-U+036F; without it ọ̀rọ̀
    splits into ọ + rọ. The common shortcut `[a-z]+` throws away far more and
    quietly changes every count in this notebook — see notebook 03b.
    """
    return re.findall(r"[\w\u0300-\u036f]+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé lórí ọ̀rọ̀ ẹ̀kọ́ — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("bare \\w+   :", re.findall(r"\w+", sample.lower()))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter; bare \\w+")
print("splits them wherever a tone mark sits on ọ or ẹ.")

## A countable question, answered

"Is *health* vocabulary distinctive to health reporting?" is countable because
both halves are in the data: a topic label and the words.

In [ ]:
def rate(texts, word):
    """Occurrences of a word per 1,000 tokens."""
    counts = Counter()
    for t in texts:
        counts.update(tokenise(t))
    total = sum(counts.values())
    return counts[word] / total * 1000 if total else 0

eng = df[df["language"] == "English"]
print(f"{'category':<16}{'docs':>6}{'rate of \'health\' per 1,000':>28}")
print("-" * 50)
for cat, g in eng.groupby("category"):
    if len(g) >= 15:
        print(f"{cat:<16}{len(g):>6}{rate(g['text'], 'health'):>28.2f}")

The rate is far higher in health reporting than anywhere else — which is
reassuring rather than interesting. A method that could not recover something
that obvious would not be trustworthy on anything subtle.

**Validate your method on a question you already know the answer to, before you
trust it on one you do not.** That is the single most useful habit in this pack.

In [ ]:
# An uncountable question, and what happens when you try.
# "Do these articles APPROVE of what they describe?"
for cat in ("politics", "health"):
    sub = df[(df["language"] == "English") & (df["category"] == cat)]
    print(f"{cat}: 'good' appears {rate(sub['text'], 'good'):.2f} per 1,000, "
          f"'bad' {rate(sub['text'], 'bad'):.2f}")
print()
print("These numbers are real and they mean almost nothing. 'Not good', 'good")
print("riddance' and a quotation of somebody else saying 'good' all count once.")
print("Counting words cannot see negation, irony, or who is speaking.")

### 🔧 Your turn 1

Pick a word central to your own research and compute its rate across the
categories.

Then write down one thing the rate *cannot* tell you about how that word is
being used. Notebook 05a gives you the tool for that second question.

## The three-part question

A question this pack can answer names all three:

1. **A corpus** — which documents, and why those
2. **A countable feature** — a word, a class of words, a length, a ratio
3. **A comparison** — against what?

"How is migration discussed in Nigerian news?" fails on 3. "Is migration
vocabulary more frequent in Hausa or Yoruba reporting on politics?" passes.

In [ ]:
# Every comparison needs a denominator. This is the commonest error in
# quantitative text work: comparing raw counts across corpora of different size.
for lang in ("Hausa", "Yoruba"):
    sub = df[df["language"] == lang]
    counts = Counter()
    for t in sub["text"]:
        counts.update(tokenise(t))
    total = sum(counts.values())
    print(f"{lang:<10} {len(sub):>5} docs   {total:>9,} tokens")
print()
print("Different totals. Any comparison must be per-1,000-tokens, never raw counts —")
print("otherwise you are measuring which corpus is bigger.")

### 🔧 Your turn 2

Write one question about this corpus that counting **can** answer and one it
**cannot**, and say which of the three parts the second one is missing.

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** A rate tells you how often a word occurs and nothing about how
it is used. It cannot distinguish approval from criticism, literal from ironic,
the author's voice from a quoted one, or a word from its negation. That is not
a limitation to apologise for — it is the boundary that makes close reading
necessary, and pointing at *where* to read closely is exactly what the count is
good for.

**Your turn 2.** A good answerable one: "Do sports articles use shorter
sentences than political ones?" — corpus, countable feature, comparison. A good
unanswerable one: "Is Nigerian journalism becoming more sensational?" — it names
no comparison and no countable feature, and this corpus has no time dimension to
support *becoming* at all.

</details>

## Where this stops

You can tell a countable question from an uncountable one. The next notebook is
about the corpus itself, and who is missing from it.